In [1]:
# ==============================================================
# COMPLETE PREPROCESSING FOR people-100.csv
# ==============================================================

# Install required libraries
%pip install pandas numpy matplotlib seaborn scikit-learn -q

# ==============================================================
# 1. IMPORT LIBRARIES
# ==============================================================
%%writefile app.py
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# ==============================================================
# 2. LOAD DATASET
# ==============================================================

FILE_PATH = "people-100.csv"

if not os.path.exists(FILE_PATH):
    available_csv_files = [
        file for file in os.listdir(".")
        if file.lower().endswith(".csv")
    ]

    raise FileNotFoundError(
        f"'{FILE_PATH}' was not found.\n"
        f"Place people-100.csv in the same folder as the notebook.\n"
        f"Available CSV files: {available_csv_files}"
    )

try:
    df = pd.read_csv(FILE_PATH)
except UnicodeDecodeError:
    df = pd.read_csv(FILE_PATH, encoding="latin-1")

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print("Original shape:", df.shape)

display(df.head())

# ==============================================================
# 3. CLEAN COLUMN NAMES
# ==============================================================

original_columns = df.columns.tolist()

df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"[\s\-]+", "_", regex=True)
      .str.replace(r"[^a-z0-9_]", "", regex=True)
      .str.replace(r"_+", "_", regex=True)
      .str.strip("_")
)

# Rename possible variations
rename_mapping = {
    "userid": "user_id",
    "user": "user_id",
    "firstname": "first_name",
    "lastname": "last_name",
    "gender": "sex",
    "dob": "date_of_birth",
    "dateofbirth": "date_of_birth",
    "birth_date": "date_of_birth",
    "jobtitle": "job_title",
    "occupation": "job_title",
    "phone_number": "phone",
    "telephone": "phone"
}

df = df.rename(
    columns={
        old: new
        for old, new in rename_mapping.items()
        if old in df.columns
    }
)

print("\nOriginal columns:")
print(original_columns)

print("\nCleaned columns:")
print(df.columns.tolist())

# ==============================================================
# 4. REMOVE UNNECESSARY COLUMNS
# ==============================================================

unnamed_columns = [
    column for column in df.columns
    if column.startswith("unnamed")
]

if unnamed_columns:
    df = df.drop(columns=unnamed_columns)
    print("\nRemoved unnamed columns:", unnamed_columns)

# Keep "index" because it may be part of the original dataset.
# Uncomment the following line if you want to remove it:
# df = df.drop(columns=["index"], errors="ignore")

# ==============================================================
# 5. REPLACE MISSING-VALUE LABELS
# ==============================================================

missing_labels = [
    "", " ", "?", "-", "--",
    "N/A", "n/a", "NA", "na",
    "NULL", "null",
    "None", "none",
    "Unknown", "unknown"
]

df = df.replace(missing_labels, np.nan)

# ==============================================================
# 6. REMOVE DUPLICATE ROWS
# ==============================================================

duplicates_before = int(df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("\nDuplicate rows removed:", duplicates_before)
print("Shape after duplicate removal:", df.shape)

# ==============================================================
# 7. CLEAN ALL TEXT COLUMNS
# ==============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

for column in text_columns:
    df[column] = (
        df[column]
          .astype("string")
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
    )

# ==============================================================
# 8. CLEAN FIRST AND LAST NAMES
# ==============================================================

for column in ["first_name", "last_name"]:
    if column in df.columns:
        df[column] = (
            df[column]
              .str.replace(r"[^A-Za-zÀ-ÖØ-öø-ÿ'\-\s]", "", regex=True)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip()
              .str.title()
        )

if "first_name" in df.columns and "last_name" in df.columns:
    df["full_name"] = (
        df["first_name"].fillna("") + " " +
        df["last_name"].fillna("")
    ).str.strip()

    df.loc[df["full_name"] == "", "full_name"] = pd.NA

# ==============================================================
# 9. CLEAN SEX/GENDER COLUMN
# ==============================================================

if "sex" in df.columns:
    df["sex"] = df["sex"].str.lower().str.strip()

    sex_mapping = {
        "m": "Male",
        "male": "Male",
        "man": "Male",
        "f": "Female",
        "female": "Female",
        "woman": "Female",
        "non-binary": "Non-binary",
        "nonbinary": "Non-binary",
        "other": "Other"
    }

    df["sex"] = df["sex"].replace(sex_mapping)

    recognised_values = [
        "Male",
        "Female",
        "Non-binary",
        "Other"
    ]

    df.loc[
        ~df["sex"].isin(recognised_values) & df["sex"].notna(),
        "sex"
    ] = "Other"

# ==============================================================
# 10. CLEAN AND VALIDATE EMAIL ADDRESSES
# ==============================================================

def clean_email(value):
    if pd.isna(value):
        return pd.NA

    email = str(value).strip().lower()
    email = re.sub(r"\s+", "", email)

    return email if email else pd.NA


def is_valid_email(value):
    if pd.isna(value):
        return False

    pattern = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"
    return bool(re.fullmatch(pattern, str(value)))


if "email" in df.columns:
    df["email"] = df["email"].apply(clean_email)
    df["email_valid"] = df["email"].apply(is_valid_email)

    df["email_domain"] = (
        df["email"]
          .str.extract(r"@([^@]+)$", expand=False)
          .str.lower()
    )

# ==============================================================
# 11. CLEAN AND VALIDATE PHONE NUMBERS
# ==============================================================

def clean_phone(value):
    if pd.isna(value):
        return pd.NA

    phone = str(value).strip()

    # Remove the common spreadsheet ".0" ending
    phone = re.sub(r"\.0$", "", phone)

    # Keep numbers and a possible leading plus sign
    phone = re.sub(r"[^\d+]", "", phone)

    # Remove plus signs that are not at the beginning
    if phone.startswith("+"):
        phone = "+" + phone[1:].replace("+", "")
    else:
        phone = phone.replace("+", "")

    return phone if phone else pd.NA


def is_valid_phone(value):
    if pd.isna(value):
        return False

    digits = re.sub(r"\D", "", str(value))
    return 7 <= len(digits) <= 15


if "phone" in df.columns:
    df["phone"] = df["phone"].apply(clean_phone)
    df["phone_valid"] = df["phone"].apply(is_valid_phone)

# ==============================================================
# 12. CLEAN USER ID
# ==============================================================

if "user_id" in df.columns:
    df["user_id"] = (
        df["user_id"]
          .astype("string")
          .str.strip()
          .str.upper()
    )

# ==============================================================
# 13. CLEAN JOB TITLES
# ==============================================================

if "job_title" in df.columns:
    df["job_title"] = (
        df["job_title"]
          .str.replace(r"\s+", " ", regex=True)
          .str.strip()
          .str.title()
    )

# ==============================================================
# 14. CONVERT DATE OF BIRTH
# ==============================================================

if "date_of_birth" in df.columns:
    # Store the original values temporarily
    original_dates = df["date_of_birth"].copy()

    # First attempt: general date conversion
    df["date_of_birth"] = pd.to_datetime(
        original_dates,
        errors="coerce"
    )

    # Second attempt for dates that may use day/month/year
    failed_dates = df["date_of_birth"].isna() & original_dates.notna()

    if failed_dates.any():
        df.loc[failed_dates, "date_of_birth"] = pd.to_datetime(
            original_dates.loc[failed_dates],
            errors="coerce",
            dayfirst=True
        )

# ==============================================================
# 15. CREATE AGE AND DATE FEATURES
# ==============================================================

if "date_of_birth" in df.columns:
    today = pd.Timestamp.today().normalize()

    year_difference = today.year - df["date_of_birth"].dt.year

    birthday_not_reached = (
        (df["date_of_birth"].dt.month > today.month) |
        (
            (df["date_of_birth"].dt.month == today.month) &
            (df["date_of_birth"].dt.day > today.day)
        )
    )

    df["age"] = year_difference - birthday_not_reached.astype("Int64")
    df["age"] = df["age"].astype("Int64")

    # Replace impossible ages with missing values
    df.loc[
        (df["age"] < 0) | (df["age"] > 120),
        "age"
    ] = pd.NA

    df["birth_year"] = df["date_of_birth"].dt.year.astype("Int64")
    df["birth_month"] = df["date_of_birth"].dt.month.astype("Int64")
    df["birth_day"] = df["date_of_birth"].dt.day.astype("Int64")
    df["birth_month_name"] = df["date_of_birth"].dt.month_name()

# ==============================================================
# 16. CREATE AGE GROUP
# ==============================================================

if "age" in df.columns:
    df["age_group"] = pd.cut(
        df["age"].astype(float),
        bins=[-1, 17, 24, 34, 44, 54, 64, 120],
        labels=[
            "Under 18",
            "18-24",
            "25-34",
            "35-44",
            "45-54",
            "55-64",
            "65+"
        ]
    )

# ==============================================================
# 17. HANDLE MISSING VALUES
# ==============================================================

# Fill selected descriptive categorical columns with "Unknown"
categorical_fill_columns = [
    "first_name",
    "last_name",
    "full_name",
    "sex",
    "job_title",
    "email_domain",
    "birth_month_name"
]

for column in categorical_fill_columns:
    if column in df.columns:
        if pd.api.types.is_categorical_dtype(df[column]):
            df[column] = df[column].astype("string")

        df[column] = df[column].fillna("Unknown")

# Keep missing emails and phone numbers as "Missing"
for column in ["email", "phone", "user_id"]:
    if column in df.columns:
        df[column] = df[column].fillna("Missing")

# Fill missing ages with the median age
if "age" in df.columns and df["age"].notna().any():
    median_age = int(round(df["age"].median()))
    df["age"] = df["age"].fillna(median_age).astype(int)

# Fill missing age groups after filling age
if "age" in df.columns:
    df["age_group"] = pd.cut(
        df["age"],
        bins=[-1, 17, 24, 34, 44, 54, 64, 120],
        labels=[
            "Under 18",
            "18-24",
            "25-34",
            "35-44",
            "45-54",
            "55-64",
            "65+"
        ]
    ).astype("string").fillna("Unknown")

# ==============================================================
# 18. REMOVE DUPLICATE EMAILS AND USER IDS
# ==============================================================

# Report duplicates without automatically deleting people
if "user_id" in df.columns:
    duplicate_user_ids = int(
        df.loc[df["user_id"] != "Missing", "user_id"].duplicated().sum()
    )
else:
    duplicate_user_ids = 0

if "email" in df.columns:
    duplicate_emails = int(
        df.loc[df["email"] != "Missing", "email"].duplicated().sum()
    )
else:
    duplicate_emails = 0

print("\nDuplicate user IDs:", duplicate_user_ids)
print("Duplicate email addresses:", duplicate_emails)

# ==============================================================
# 19. DATA QUALITY REPORT
# ==============================================================

quality_information = {
    "Total rows": len(df),
    "Total columns": len(df.columns),
    "Duplicate complete rows": int(df.duplicated().sum()),
    "Duplicate user IDs": duplicate_user_ids,
    "Duplicate email addresses": duplicate_emails,
    "Total remaining missing values": int(df.isna().sum().sum())
}

if "email_valid" in df.columns:
    quality_information["Valid emails"] = int(df["email_valid"].sum())
    quality_information["Invalid or missing emails"] = int(
        (~df["email_valid"]).sum()
    )

if "phone_valid" in df.columns:
    quality_information["Valid phone numbers"] = int(
        df["phone_valid"].sum()
    )
    quality_information["Invalid or missing phone numbers"] = int(
        (~df["phone_valid"]).sum()
    )

quality_report = pd.DataFrame(
    list(quality_information.items()),
    columns=["Quality measure", "Value"]
)

print("\n" + "=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

display(quality_report)

# ==============================================================
# 20. MISSING VALUES REPORT
# ==============================================================

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing values": df.isna().sum().values,
    "Missing percentage": (
        df.isna().mean().values * 100
    ).round(2)
})

print("\nMissing-values report:")
display(missing_report)

# ==============================================================
# 21. VISUALISE SEX DISTRIBUTION
# ==============================================================

if "sex" in df.columns:
    plt.figure(figsize=(8, 5))

    sns.countplot(
        data=df,
        x="sex",
        hue="sex",
        palette="Set2",
        legend=False
    )

    plt.title("Distribution by Sex")
    plt.xlabel("Sex")
    plt.ylabel("Number of People")
    plt.tight_layout()
    plt.show()

# ==============================================================
# 22. VISUALISE AGE DISTRIBUTION
# ==============================================================

if "age" in df.columns:
    plt.figure(figsize=(10, 5))

    sns.histplot(
        data=df,
        x="age",
        bins=15,
        kde=True,
        color="royalblue"
    )

    plt.title("Age Distribution")
    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# ==============================================================
# 23. VISUALISE AGE GROUP DISTRIBUTION
# ==============================================================

if "age_group" in df.columns:
    age_order = [
        "Under 18",
        "18-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+",
        "Unknown"
    ]

    present_groups = [
        group for group in age_order
        if group in df["age_group"].unique()
    ]

    plt.figure(figsize=(10, 5))

    sns.countplot(
        data=df,
        x="age_group",
        order=present_groups,
        hue="age_group",
        palette="viridis",
        legend=False
    )

    plt.title("Age Group Distribution")
    plt.xlabel("Age Group")
    plt.ylabel("Number of People")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

# ==============================================================
# 24. VISUALISE TOP EMAIL DOMAINS
# ==============================================================

if "email_domain" in df.columns:
    top_domains = (
        df.loc[df["email_domain"] != "Unknown", "email_domain"]
          .value_counts()
          .head(10)
          .reset_index()
    )

    top_domains.columns = ["Email domain", "Count"]

    if not top_domains.empty:
        plt.figure(figsize=(10, 5))

        sns.barplot(
            data=top_domains,
            x="Count",
            y="Email domain",
            hue="Email domain",
            palette="mako",
            legend=False
        )

        plt.title("Top 10 Email Domains")
        plt.tight_layout()
        plt.show()

# ==============================================================
# 25. CREATE MACHINE-LEARNING-READY DATA
# ==============================================================

ml_df = df.copy()

# Remove identifier and personal-information columns because they
# normally should not be used as machine-learning input features.
columns_not_for_ml = [
    "index",
    "user_id",
    "first_name",
    "last_name",
    "full_name",
    "email",
    "phone",
    "date_of_birth"
]

ml_df = ml_df.drop(
    columns=[
        column for column in columns_not_for_ml
        if column in ml_df.columns
    ],
    errors="ignore"
)

# Convert Boolean columns to integers
boolean_columns = ml_df.select_dtypes(include=["bool"]).columns.tolist()

for column in boolean_columns:
    ml_df[column] = ml_df[column].astype(int)

# Separate numerical and categorical columns
numerical_columns = ml_df.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = ml_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nNumerical ML columns:")
print(numerical_columns)

print("\nCategorical ML columns:")
print(categorical_columns)

# Numerical preprocessing:
# - replace missing values with the median
# - standardise values
numerical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# Categorical preprocessing:
# - replace missing values with the most frequent value
# - use one-hot encoding
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    # Support older versions of scikit-learn
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        encoder
    )
])

transformers = []

if numerical_columns:
    transformers.append(
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        )
    )

if categorical_columns:
    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    )

if transformers:
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    processed_array = preprocessor.fit_transform(ml_df)

    try:
        processed_feature_names = (
            preprocessor.get_feature_names_out()
        )
    except AttributeError:
        processed_feature_names = [
            f"feature_{number}"
            for number in range(processed_array.shape[1])
        ]

    ml_ready_df = pd.DataFrame(
        processed_array,
        columns=processed_feature_names
    )
else:
    ml_ready_df = pd.DataFrame()

# ==============================================================
# 26. DISPLAY FINAL RESULTS
# ==============================================================

print("\n" + "=" * 60)
print("CLEANED DATASET")
print("=" * 60)

print("Cleaned dataset shape:", df.shape)
display(df.head(10))

print("\n" + "=" * 60)
print("MACHINE-LEARNING-READY DATASET")
print("=" * 60)

print("ML-ready dataset shape:", ml_ready_df.shape)
display(ml_ready_df.head(10))

# ==============================================================
# 27. SAVE RESULTS
# ==============================================================

cleaned_file = "people-100-cleaned.csv"
ml_file = "people-100-ml-ready.csv"
report_file = "people-100-quality-report.csv"

df.to_csv(cleaned_file, index=False)
ml_ready_df.to_csv(ml_file, index=False)
quality_report.to_csv(report_file, index=False)

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Created files:")
print(f"1. {cleaned_file}")
print(f"2. {ml_file}")
print(f"3. {report_file}")

Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%%writefile` not found.
